# 04 · Training: two-stage LightGBM + decision tuning

**Folds** — 5 folds grouped by S1 id (deterministic hash), so all candidates of one S1 share a fold.

**Stage 1** — LightGBM binary classifier on the pair features. Fold models are fit on a sample of S1 entities (`train_frac`), then **every** train pair gets an out-of-fold probability `p1`. Scoring every pair (not just a sample) matters: target-competition in stage 2 needs the p1 of *all* S1s that compete for a record.

**Stage 2** — LightGBM on stage-1 features + `p1` + set context: the S1's p1 max/second/sum/rank, the best p1 any *other* S1 gives the target, and similarity of the candidate to the S1's top-1 / top-2 candidates (true matches are noisy copies of each other). Out-of-fold `p` for every pair.

**Decision** — grid on OOF scores with the exact macro F0.5: plain threshold vs. exclusivity + threshold vs. exclusivity + per-S1 expected-F0.5 subset selection. The winner is frozen in `artifacts/models/decision.json`.

Compute notes: defaults fit a 64 GB+ machine (stage 1 fits on ~40M pairs). On smaller machines lower `train_frac`; on bigger ones raise it. It only changes the rows used to *fit*; OOF scoring always covers every pair.

In [ ]:
# --- Setup: make src/ importable, load run settings -------------------------
import os, sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "entity_forge").is_dir())
sys.path.insert(0, str(ROOT / "src"))

# Override settings here or with EF_* environment variables before starting Jupyter.
# os.environ["EF_DEV_MODE"] = "1"     # small consistent slice: end-to-end smoke test on a laptop
# os.environ["EF_N_THREADS"] = "32"

import polars as pl
from entity_forge import stages
from entity_forge.settings import Settings

pl.Config.set_tbl_rows(30); pl.Config.set_fmt_str_lengths(80); pl.Config.set_tbl_width_chars(220)
stages.setup_logging()
S = Settings.from_env()
print(f"root={S.root}\nwork_dir={S.work_dir}\ndev_mode={S.dev_mode} threads={S.n_threads}")

## Stage 1

In [ ]:
# train_frac = share of S1 entities used to FIT the fold models (OOF scoring always covers all).
# Full data: 0.3 ≈ 40M pairs. Raise it on a bigger box if memory/time allow.
scores1 = stages.run_stage1_cv(S, train_frac=0.3 if not S.dev_mode else 1.0)
scores1.group_by("label").agg(pl.col("p1").mean(), pl.len())

In [ ]:
imp = pl.concat([pl.read_csv(S.reports / f"importance_stage1_fold{f}.csv") for f in range(S.n_folds)])
imp.group_by("feature").agg(pl.col("gain").mean()).sort("gain", descending=True).head(25)

## Stage 2

In [ ]:
scores2 = stages.run_stage2_cv(S, train_frac=0.4 if not S.dev_mode else 1.0)
scores2.group_by("label").agg(pl.col("p1", "p").mean(), pl.len())

## Decision rule (exact metric, out-of-fold)

In [ ]:
best_s1, grid_s1 = stages.run_decision_tuning(S, score_col="p1")   # stage-1 only, for the ablation table
best, grid = stages.run_decision_tuning(S, score_col="p")           # final (written last -> frozen)
print("stage-1 best:", best_s1)
print("stage-2 best:", best)
grid.head(15)

## Per-country out-of-fold results

In [ ]:
stages.oof_report(S)

## Error analysis (OOF): wrong merges and missed matches

In [ ]:
false_pos, false_neg = stages.error_examples(S, n=15)
print("FALSE POSITIVES (wrong merges)"); display(false_pos)
print("FALSE NEGATIVES (missed; p = null means lost in blocking)"); display(false_neg)

## Calibration (OOF stage-2)

In [ ]:
(scores2.with_columns((pl.col("p") * 10).floor().clip(0, 9).alias("bin"))
        .group_by("bin").agg(pl.col("p").mean().alias("mean_p"), pl.col("label").mean().alias("hit_rate"), pl.len())
        .sort("bin"))